# Predict letter using MediaPipe's HandLandmarker and XGBoost Classifier

In [1]:
%load_ext autoreload
%autoreload 2

from handspell.model_landmarks import create_landmarker
# load landmarker
landmarker = create_landmarker()

MediaPipe model: ../models/hand_landmarker.task


I0000 00:00:1792535693.149757 6967765 init-domain.cc:134] Fiber init: default domain = pthread, concurrency = 8, prefix = pthread-default
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1792535693.156484 6967769 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792535693.161340 6967771 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [2]:
from handspell.model_landmarks import load_asl_model
model = load_asl_model('../models/handspell_xgboost_megadata.joblib')

In [3]:
model

,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",80
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.d

In [5]:
import numpy as np
from joblib import load

from handspell.model_landmarks import read_rgb, get_features
from handspell.model_landmarks import create_landmarker
# from handspell.utils import LETTER_TO_INDEX

LETTERS = "ABCDEFGHIKLMNOPQRSTUVWXY"
LABEL_MAP = {letter: index for index, letter in enumerate(LETTERS)}

# load landmarker
landmarker = create_landmarker()

# Load ASL model
model = load('../models/handspell_xgboost_megadata.joblib')

image_path = "../data/raw_data/A05_notFonz_original/images/Afonso/H0.jpg"
# image_path = "../data/raw_data/A04_mandalay_aslhg/ASL_HG_36000/asl_dataset/E/P1_E_1.jpg"
# image_path = "../data/raw_data/A01_kaggle_aslmnist/train/E/E_44.jpg"
# image_path = "/Users/delphinepessoa/Downloads/Q-test.jpeg"

# image_path = "../data/dataset/D01_megadataset/B/A03_kaggle_asl__1.jpg"
r = get_features(read_rgb(image_path), landmarker)
print(r)
features = np.expand_dims(r[0], 0)
usable = np.isfinite(features).all()
print(usable)
model.predict(features)
list(zip(list(LETTERS), list([float(el) for el in model.predict_proba(features)[0]])))

MediaPipe model: ../models/hand_landmarker.task


W0000 00:00:1792535845.381735 6969246 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792535845.386579 6969242 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


(array([ 0.        ,  0.        ,  0.        ,  0.2139704 , -0.42509186,
        0.08064908,  0.62270474, -0.60241777,  0.00744619,  1.0273468 ,
       -0.561213  , -0.07494963,  1.3173085 , -0.45481342, -0.15633169,
        0.66617566, -0.8136715 , -0.36133647,  1.2781476 , -0.82690626,
       -0.5146412 ,  1.643767  , -0.799103  , -0.578094  ,  1.9233304 ,
       -0.7643614 , -0.6059148 ,  0.6770613 , -0.55939597, -0.47818825,
        1.3854268 , -0.58360624, -0.619017  ,  1.7983671 , -0.56351733,
       -0.6396904 ,  2.108137  , -0.5527888 , -0.65644634,  0.6936206 ,
       -0.24737442, -0.5537929 ,  1.3733475 , -0.21309099, -0.6148325 ,
        1.2730727 , -0.19538394, -0.46439242,  1.0920141 , -0.208645  ,
       -0.35382324,  0.7042348 ,  0.0845993 , -0.62275845,  1.2302889 ,
        0.06763229, -0.5995076 ,  1.1282923 ,  0.05092274, -0.42969498,
        0.95075166,  0.03877151, -0.29956406], dtype=float32), [NormalizedLandmark(x=0.172604501247406, y=0.7161299586296082, z=1.25275

[('A', 0.01549573801457882),
 ('B', 0.0019952638540416956),
 ('C', 0.0915970653295517),
 ('D', 0.006422181148082018),
 ('E', 0.0019169901497662067),
 ('F', 0.002230365527793765),
 ('G', 0.2603219151496887),
 ('H', 0.5163927674293518),
 ('I', 0.002555853920057416),
 ('K', 0.0016544228419661522),
 ('L', 0.002078555291518569),
 ('M', 0.009712121449410915),
 ('N', 0.06031981110572815),
 ('O', 0.011083298362791538),
 ('P', 0.0013727025361731648),
 ('Q', 0.0009284017141908407),
 ('R', 0.0010823631891980767),
 ('S', 0.0015154288848862052),
 ('T', 0.004023701883852482),
 ('U', 0.0009858958655968308),
 ('V', 0.0013087656116113067),
 ('W', 0.0009073494584299624),
 ('X', 0.002870165975764394),
 ('Y', 0.0012289404403418303)]

In [58]:
r[0] is None

True

In [59]:
len(r[0])

TypeError: object of type 'NoneType' has no len()

In [54]:
type(model)

xgboost.sklearn.XGBClassifier

In [55]:
type(landmarker)

mediapipe.tasks.python.vision.hand_landmarker.HandLandmarker

In [1]:
from handspell.model_landmarks import preprocess_and_predict_api, create_landmarker, load_asl_model

image_path = "../data/raw_data/A05_notFonz_original/images/Afonso/H0.jpg"
# image_path = "../data/mapping/ASL Letter Reference/N.jpg"

landmarker = create_landmarker('../models/hand_landmarker.task', download_if_needed=False)
asl_model = load_asl_model('../models/handspell_xgboost_megadata.joblib')
r = preprocess_and_predict_api(image_path, landmarker, asl_model)
r

I0000 00:00:1792574941.929610 7158687 init-domain.cc:134] Fiber init: default domain = pthread, concurrency = 8, prefix = pthread-default
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1792574941.940016 7158694 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792574941.946204 7158694 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792574942.268613 7158689 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


{'letter': 'H',
 'confidence': 0.5163927674293518,
 'scores': [0.01549573801457882,
  0.0019952638540416956,
  0.0915970653295517,
  0.006422181148082018,
  0.0019169901497662067,
  0.002230365527793765,
  0.2603219151496887,
  0.5163927674293518,
  0.002555853920057416,
  0.0016544228419661522,
  0.002078555291518569,
  0.009712121449410915,
  0.06031981110572815,
  0.011083298362791538,
  0.0013727025361731648,
  0.0009284017141908407,
  0.0010823631891980767,
  0.0015154288848862052,
  0.004023701883852482,
  0.0009858958655968308,
  0.0013087656116113067,
  0.0009073494584299624,
  0.002870165975764394,
  0.0012289404403418303],
 'features': [0.0,
  0.0,
  0.0,
  0.2139703929424286,
  -0.42509186267852783,
  0.08064907789230347,
  0.6227047443389893,
  -0.6024177670478821,
  0.007446190342307091,
  1.0273468494415283,
  -0.5612130165100098,
  -0.07494962960481644,
  1.3173085451126099,
  -0.4548134207725525,
  -0.15633168816566467,
  0.6661756634712219,
  -0.8136715292930603,
  -0.

In [4]:
import numpy as np
api_features = np.array([
    0,
    0,
    0,
    -0.42293864488601685,
    -0.21675117313861847,
    0.04182647913694382,
    -0.5919657945632935,
    -0.6486372351646423,
    -0.034418169409036636,
    -0.5172063112258911,
    -1.0708417892456055,
    -0.11646294593811035,
    -0.3897912800312042,
    -1.36362624168396,
    -0.19378481805324554,
    -0.8138449788093567,
    -0.6875075101852417,
    -0.33513811230659485,
    -0.8348287343978882,
    -1.2942681312561035,
    -0.4836176931858063,
    -0.8110087513923645,
    -1.6566534042358398,
    -0.5437530875205994,
    -0.7790319919586182,
    -1.9385535717010498,
    -0.5698471665382385,
    -0.560249924659729,
    -0.7035452723503113,
    -0.43720024824142456,
    -0.5913792252540588,
    -1.4090611934661865,
    -0.5758127570152283,
    -0.5687268972396851,
    -1.8260786533355713,
    -0.6050917506217957,
    -0.5531186461448669,
    -2.153700828552246,
    -0.6266191601753235,
    -0.24913108348846436,
    -0.7196990251541138,
    -0.5065061450004578,
    -0.21124747395515442,
    -1.3887029886245728,
    -0.5987398028373718,
    -0.19365736842155457,
    -1.2611911296844482,
    -0.480118989944458,
    -0.20770247280597687,
    -1.0677367448806763,
    -0.3804255723953247,
    0.07997166365385056,
    -0.7288329601287842,
    -0.5708609223365784,
    0.05851652845740318,
    -1.2516131401062012,
    -0.5853148698806763,
    0.048149824142456055,
    -1.1282093524932861,
    -0.44398367404937744,
    0.04143413528800011,
    -0.9417948722839355,
    -0.3276195824146271
  ])

In [5]:
api_features

array([ 0.        ,  0.        ,  0.        , -0.42293864, -0.21675117,
        0.04182648, -0.59196579, -0.64863724, -0.03441817, -0.51720631,
       -1.07084179, -0.11646295, -0.38979128, -1.36362624, -0.19378482,
       -0.81384498, -0.68750751, -0.33513811, -0.83482873, -1.29426813,
       -0.48361769, -0.81100875, -1.6566534 , -0.54375309, -0.77903199,
       -1.93855357, -0.56984717, -0.56024992, -0.70354527, -0.43720025,
       -0.59137923, -1.40906119, -0.57581276, -0.5687269 , -1.82607865,
       -0.60509175, -0.55311865, -2.15370083, -0.62661916, -0.24913108,
       -0.71969903, -0.50650615, -0.21124747, -1.38870299, -0.5987398 ,
       -0.19365737, -1.26119113, -0.48011899, -0.20770247, -1.06773674,
       -0.38042557,  0.07997166, -0.72883296, -0.57086092,  0.05851653,
       -1.25161314, -0.58531487,  0.04814982, -1.12820935, -0.44398367,
        0.04143414, -0.94179487, -0.32761958])

In [6]:
np.array(r["features"])

array([ 0.        ,  0.        ,  0.        ,  0.21397039, -0.42509186,
        0.08064908,  0.62270474, -0.60241777,  0.00744619,  1.02734685,
       -0.56121302, -0.07494963,  1.31730855, -0.45481342, -0.15633169,
        0.66617566, -0.81367153, -0.36133647,  1.27814758, -0.82690626,
       -0.51464123,  1.643767  , -0.79910302, -0.57809401,  1.92333043,
       -0.76436138, -0.60591477,  0.67706132, -0.55939597, -0.47818825,
        1.38542676, -0.58360624, -0.619017  ,  1.79836714, -0.56351733,
       -0.6396904 ,  2.10813689, -0.55278879, -0.65644634,  0.69362062,
       -0.24737442, -0.55379289,  1.37334752, -0.21309099, -0.61483252,
        1.27307272, -0.19538394, -0.46439242,  1.09201407, -0.208645  ,
       -0.35382324,  0.70423478,  0.0845993 , -0.62275845,  1.23028886,
        0.06763229, -0.59950757,  1.12829232,  0.05092274, -0.42969498,
        0.95075166,  0.03877151, -0.29956406])